# 14 · The post-filter on a real stream

Notebook 13 found one hint worth its bits: a post-filter the host fits, per class of LR pixel,
on notebook 4's output - +0.0066 SSIM-Y and +0.87 dB luma for 6.2 kbit an update, where the same
bits spent on H.264 would buy a tenth of that. But it measured on patches, fitted on a source's
other frames, and priced the bits with an exchange rate estimated across sources. This notebook
asks whether the post-filter still adds anything on a real stream:

- **Whole frames.** The host fits on the whole 720p frame it sends, and the client filters the
  whole frame its model made.
- **Over time.** The host fits on a frame and sends the filters with it - every frame, every
  5th, or once a clip - or, adding no delay at all, fits on a frame and the client uses the
  filters on the next one.
- **At the same total bitrate.** Every clip is streamed at five bitrates, and the H.264 stream's
  real coded size is counted. The hinted stream - video plus filters - is set against the plain
  stream at the same total bits, read off the plain stream's own curve. No exchange rate.

    host:   HR frame ──► stream ──► decode its own stream ──► notebook 4's model ──► fit 25 filters against the HR ──► send
    client: LR frame ──► notebook 4's model ──► classes from the LR ──► each pixel through its class's filter ──► picture

Notebook 4's model runs in torch, the same weights as its published graph; the post-filter is
`helpers/postfilter.py`, as notebook 13 measures it. Nothing is trained.

Steps:

1. **Clips** - validation sources rebuilt whole, streamed the way the host sends today.
2. **The client** - notebook 4's model, and the zero-bit post-filter it carries.
3. **The host's fits** - every frame of every clip at every bitrate, and every schedule scored.
4. **Over time** - at the default bitrate: what each schedule adds, and what it costs a second.
5. **At the same bits** - each schedule against the plain stream at the same total.

Needs: `00_data_preprocess.ipynb`, the downloaded datasets, and `checkpoints/upscale_strategies.pt`
from `04_upscale_strategies.ipynb`. Writes nothing.

In [1]:
"""Settings."""

import numpy as np
import torch
from torch.utils.data import DataLoader, Subset

from helpers import degrade, metrics, models, motion, postfilter, sources, training
from helpers.patches import PatchPairs
from helpers.postfilter import luma
from helpers.training import DATA_DIR, DEVICE

MODEL = training.CHECKPOINT_DIR / "upscale_strategies.pt"   # notebook 4's model: the client's
SOURCES_PER_KIND = 8                                       # validation sources of each kind, rebuilt as whole clips
BITRATES = (0.15, 0.25, 0.46, 0.7, 1.0)                    # bits per LR pixel; 0.46 is the host's default
PATH = {"filter": "bicubic", "deblock": False}             # what the host sends today: swscale's bicubic, no loop filter (notebook 8)
SCHEDULES = {   # name -> the frame whose fit frame t uses (None: the zero-bit filters)
    "every frame": lambda t: t,
    "every frame, a frame late": lambda t: t - 1 if t else None,
    "every 5th frame": lambda t: t - t % 5,
    "once a clip": lambda t: 0,
}

TAPS, LEVELS = 5, 5                 # as notebook 13: 5x5 filters, 25 classes
RIDGE = 1.0                         # a fit is pulled toward the zero-bit filter with the weight of one pixel
MIN_PIXELS = 2000                   # a class a frame shows less often keeps the zero-bit filter
COEFFICIENT_BITS, STEP_BITS = 10, 16
TRAIN_STEP = 2                      # every 2nd training patch fits the zero-bit filter

CLASSES = postfilter.DIRECTIONS * LEVELS
DEFAULT_BPP = degrade.STREAM_BPP
KINDS = ("screen", "photo", "video")
training.seed_everything(0)

## 1. Clips

Validation sources the model never trained on, `SOURCES_PER_KIND` of each kind, spread across the
split, rebuilt whole: a screen or a photo as the 1280x720 window that moved over it (the motion
`00_data_preprocess.ipynb` recorded), a video as every frame it has. Each clip is streamed the way
the host sends today at every bitrate, and the H.264 stream's size is kept: x264 holds a constant
bitrate only as a ceiling, so a clip that does not need the bits - a still screen - comes in under.

In [2]:
manifest = training.load_manifest()
SCALE, HALO, CORE = manifest["scale"], manifest["halo"], manifest["patch_lr"]
val_sources = manifest["splits"]["val"]["sources"]


def content(source):
    """What a source shows: a drawn desktop screen, a photo (as a moving clip) or a video."""
    if source["dataset"] == "screens":
        return "screen"
    return "photo" if "motion" in source else "video"


def clip_of(source):
    """A validation source whole, as HR frames (T, h, w, 3) and its frame rate: an image as the
    window that moved over it, a video as every frame it has."""
    path = sources.RAW / source["dataset"] / source["file"]
    if source["dataset"] in sources.SEQUENCES:
        return sources.load_sequence(path)
    shown = motion.Motion(**{**source["motion"], "window": tuple(source["motion"]["window"])})
    return motion.render(sources.load_image(path), shown), manifest["image_clip"]["fps"]


chosen = []
for kind in KINDS:
    of_kind = [source for source in val_sources if content(source) == kind]
    chosen += of_kind[::len(of_kind) // SOURCES_PER_KIND][:SOURCES_PER_KIND]

clips = []   # per source: what it shows, its HR frames and frame rate, and per bitrate its LR frames and coded bits
for source in chosen:
    frames, fps = clip_of(source)
    streams = {}
    for bpp in BITRATES:
        hr, lr, bits = degrade.degrade(frames, degrade.Degradation(scale=SCALE, bpp=bpp, fps=fps, **PATH), with_bits=True)
        streams[bpp] = (lr, bits)
    clips.append({"kind": content(source), "file": source["file"], "hr": hr, "fps": fps, "streams": streams})

print(f"{'':<8}{'clips':>6}{'frames':>8}{'LR':>10}   coded bits per LR pixel, at a target of")
print(f"{'':<32}" + "".join(f"{bpp:>8}" for bpp in BITRATES))
for kind in KINDS:
    of_kind = [clip for clip in clips if clip["kind"] == kind]
    lr_frames = of_kind[0]["streams"][BITRATES[0]][0]
    coded = [np.mean([clip["streams"][bpp][1] / clip["streams"][bpp][0][..., 0].size for clip in of_kind]) for bpp in BITRATES]
    print(f"{kind:<8}{len(of_kind):6}{np.mean([len(clip['hr']) for clip in of_kind]):8.0f}"
          f"{f'{lr_frames.shape[2]}x{lr_frames.shape[1]}':>10}{'':<4}" + "".join(f"{value:8.3f}" for value in coded))

         clips  frames        LR   coded bits per LR pixel, at a target of
                                    0.15    0.25    0.46     0.7     1.0
screen       8      30   640x360       0.104   0.156   0.268   0.365   0.475
photo        8      30   640x360       0.118   0.195   0.354   0.498   0.642
video        8      32   640x360       0.129   0.218   0.394   0.591   0.820


## 2. The client

Notebook 4's model, and the zero-bit post-filter: 25 filters fitted once on every 2nd training patch,
exactly as in notebook 13, and built into the client. A host's filters are sent as differences
from these, and a class the host does not send keeps them.

In [3]:
checkpoint = training.load_checkpoint(MODEL)
model = getattr(models, checkpoint["architecture"])(scale=SCALE, **checkpoint["arguments"]).to(DEVICE).eval()
model.load_state_dict(checkpoint["state_dict"])
trim = training.trimmer(manifest)


def no_group(n):
    """Group 0 for all `n` pictures."""
    return torch.zeros(n, dtype=torch.long, device=DEVICE)


train_set = PatchPairs(DATA_DIR, "train", manifest, augment=False)
train_index = np.arange(0, len(train_set), TRAIN_STEP)
with torch.no_grad():
    sample = torch.cat([luma(lr.to(DEVICE))[..., HALO:HALO + CORE, HALO:HALO + CORE]
                        for lr, _ in DataLoader(Subset(train_set, train_index[::8]), batch_size=256)])
    levels = postfilter.activity_levels(sample, LEVELS)
    general = postfilter.Fit(1, CLASSES, 1, TAPS, DEVICE)
    for lr, hr in DataLoader(Subset(train_set, train_index), batch_size=128):
        lr, hr = lr.to(DEVICE), hr.to(DEVICE)
        pixel_class = postfilter.classes(luma(lr), levels)[:, HALO:HALO + CORE, HALO:HALO + CORE]
        general.add(postfilter.windows(luma(trim(model(lr))), TAPS), luma(hr).flatten(2),
                    postfilter.upsampled(pixel_class, SCALE).flatten(1), no_group(len(lr)))
zero = general.solve()   # (1, classes, 26, 1)
print(f"notebook 4's model: its {checkpoint['strategy']}; the zero-bit post-filter from {len(train_index):,} training patches")

notebook 4's model: its luma 24 + 2 blocks; the zero-bit post-filter from 8,570 training patches


## 3. The host's fits

Every frame of every clip at every bitrate: notebook 4's model on the decoded LR frame, and that
frame's 25 filters fitted against its HR frame - pulled toward the zero-bit filters, rounded to
what the client would receive, a class seen on fewer than 2,000 pixels left at the zero-bit filter.
Then every schedule scored on every frame (luma SSIM and PSNR over the whole 720p frame), and its
bits counted: every update it sends, at the size its frame's fit came to.

In [4]:
def to_tensor(images):
    """uint8 (N, h, w, 3) -> float (N, 3, h, w) in [0, 1] on DEVICE."""
    return torch.from_numpy(np.ascontiguousarray(images)).to(DEVICE).permute(0, 3, 1, 2).float() / 255


records = []   # one per clip, bitrate and schedule
with torch.no_grad():
    for clip in clips:
        frames = len(clip["hr"])
        for bpp, (lr_frames, video_bits) in clip["streams"].items():
            fit = postfilter.Fit(frames, CLASSES, 1, TAPS, DEVICE)
            pictures, targets, pixel_classes = [], [], []
            for t in range(frames):   # frame by frame: a 720p frame is 921,600 windows of 26 values
                lr, hr = to_tensor(lr_frames[t:t + 1]), to_tensor(clip["hr"][t:t + 1])
                picture = model(lr)
                pixel_class = postfilter.upsampled(postfilter.classes(luma(lr), levels), SCALE)
                fit.add(postfilter.windows(luma(picture), TAPS), luma(hr).flatten(2), pixel_class.flatten(1),
                        torch.full((1,), t, device=DEVICE))
                pictures.append(picture)
                targets.append(hr)
                pixel_classes.append(pixel_class)
            sent = fit.pixels() >= MIN_PIXELS
            received = torch.where(sent[..., None, None],
                                   postfilter.quantise(fit.solve(prior=zero, ridge=RIDGE), zero, COEFFICIENT_BITS), zero)
            bank = torch.cat([received, zero]).float()   # a frame's filters; the last: the zero-bit ones
            update = postfilter.update_bits(sent, 1, TAPS, COEFFICIENT_BITS, STEP_BITS).tolist()

            uses = {"model alone": None, "zero bits": [frames] * frames}
            for name, rule in SCHEDULES.items():
                uses[name] = [frames if rule(t) is None else rule(t) for t in range(frames)]
            for name, use in uses.items():
                scores = []
                for t in range(frames):
                    picture = pictures[t] if use is None else postfilter.post_filter(
                        pictures[t], bank, pixel_classes[t], torch.full((1,), use[t], device=DEVICE), TAPS)
                    scores.append(metrics.per_image(picture, targets[t]))
                hint_bits = 0 if use is None else sum(update[f] for f in set(use) if f < frames)
                records.append({"kind": clip["kind"], "file": clip["file"], "bpp": bpp, "schedule": name,
                                "ssim": float(torch.cat([s["ssim"] for s in scores]).mean()),
                                "y": float(torch.cat([s["y"] for s in scores]).mean()),
                                "video_bits": video_bits, "hint_bits": hint_bits, "seconds": frames / clip["fps"]})
print(f"{len(records):,} scores: {len(clips)} clips x {len(BITRATES)} bitrates x {len(SCHEDULES) + 2} ways to run the client; "
      f"an update is {np.mean(update) / 1e3:.1f} kbit on the last clip's last bitrate")

720 scores: 24 clips x 5 bitrates x 6 ways to run the client; an update is 5.2 kbit on the last clip's last bitrate


## 4. Over time

At the host's default bitrate: each schedule against the zero-bit post-filter - the client without
hints - clip by clip, then averaged over the clips of a kind. Its cost is its bits a second, and
their share of the video's own.

In [5]:
def pick(**match):
    """The records that match, in clip order."""
    return [r for r in records if all(r[key] == value for key, value in match.items())]


print(f"{'':<34}{'ssim-y':>8}{'gain':>9}{'psnr-y':>8}{'gain':>7}{'kbit/s':>9}{'of the video':>14}")
for kind in KINDS:
    base = pick(kind=kind, bpp=DEFAULT_BPP, schedule="zero bits")
    video_rate = np.mean([r["video_bits"] / r["seconds"] for r in base])
    print(f"{kind}: the video at {video_rate / 1e3:,.0f} kbit/s")
    for name in ("model alone", "zero bits", *SCHEDULES):
        rows = pick(kind=kind, bpp=DEFAULT_BPP, schedule=name)
        ssim, y = np.mean([r["ssim"] for r in rows]), np.mean([r["y"] for r in rows])
        rate = np.mean([r["hint_bits"] / r["seconds"] for r in rows])
        print(f"  {name:<32}{ssim:8.4f}{ssim - np.mean([r['ssim'] for r in base]):+9.4f}{y:8.2f}"
              f"{y - np.mean([r['y'] for r in base]):+7.2f}{rate / 1e3:9.1f}{rate / video_rate:14.1%}")

                                    ssim-y     gain  psnr-y   gain   kbit/s  of the video
screen: the video at 1,855 kbit/s
  model alone                       0.9482  -0.0011   28.94  -0.17      0.0          0.0%
  zero bits                         0.9494  +0.0000   29.11  +0.00      0.0          0.0%
  every frame                       0.9546  +0.0053   29.54  +0.43    150.7          8.1%
  every frame, a frame late         0.9541  +0.0047   29.49  +0.38    145.7          7.9%
  every 5th frame                   0.9542  +0.0049   29.51  +0.40     29.9          1.6%
  once a clip                       0.9451  -0.0043   29.45  +0.34      4.6          0.2%
photo: the video at 2,449 kbit/s
  model alone                       0.8597  -0.0005   29.21  -0.24      0.0          0.0%
  zero bits                         0.8602  +0.0000   29.45  +0.00      0.0          0.0%
  every frame                       0.8629  +0.0027   29.81  +0.36    158.5          6.5%
  every frame, a frame late      

## 5. At the same bits

Per clip, the plain stream - the client with its zero-bit post-filter - at five bitrates is a
curve of quality against the bits it really took. A hinted stream at a bitrate takes the video's
bits plus the filters'; the plain stream's quality at that same total is read off its curve
(linear in log bits, between the two bitrates around it), and the difference is what the filters
add at equal bits. A total above the clip's highest bitrate has nothing to compare with and is
left out. Averaged over the clips of a kind, as SSIM-Y / dB luma.

In [6]:
def at_equal_bits(kind, schedule, bpp, key):
    """Mean over the clips of `kind`: the hinted stream's score minus the plain stream's at the same total bits."""
    gains = []
    for clip in [c for c in clips if c["kind"] == kind]:
        plain = sorted((r["video_bits"], r[key]) for r in pick(file=clip["file"], schedule="zero bits"))
        hinted = pick(file=clip["file"], schedule=schedule, bpp=bpp)[0]
        total = hinted["video_bits"] + hinted["hint_bits"]
        if total <= plain[-1][0]:
            bits, quality = zip(*plain)
            gains.append(hinted[key] - np.interp(np.log(total), np.log(bits), quality))
    return np.mean(gains) if gains else float("nan"), len(gains)


print(f"{'':<34}" + "".join(f"{bpp:>16}" for bpp in BITRATES[:-1]) + "   (target bits per LR pixel)")
for kind in KINDS:
    print(kind)
    for name in SCHEDULES:
        cells = []
        for bpp in BITRATES[:-1]:
            (ssim, clips_compared), (db, _) = at_equal_bits(kind, name, bpp, "ssim"), at_equal_bits(kind, name, bpp, "y")
            cells.append(f"{ssim:+.4f}/{db:+.2f}" + ("" if clips_compared == SOURCES_PER_KIND else f"({clips_compared})"))
        print(f"  {name:<32}" + "".join(f"{cell:>16}" for cell in cells))
print("\nSSIM-Y / dB luma over the plain stream at the same total bits; (n): only n clips had a plain stream to compare with")

                                              0.15            0.25            0.46             0.7   (target bits per LR pixel)
screen
  every frame                     -0.0010/+0.06(5)+0.0036/+0.26(5)+0.0053/+0.38(5)+0.0064/+0.38(4)
  every frame, a frame late       -0.0020/+0.01(5)+0.0027/+0.20(5)+0.0046/+0.31(5)+0.0059/+0.29(4)
  every 5th frame                 +0.0031/+0.26(7)+0.0051/+0.33(6)+0.0063/+0.36(5)+0.0068/+0.44(7)
  once a clip                        -0.0057/+0.01-0.0078/+0.19(6)-0.0047/+0.35(7)+0.0034/+0.40(7)
photo
  every frame                        -0.0102/-0.03   -0.0026/+0.19   -0.0010/+0.26+0.0000/+0.24(7)
  every frame, a frame late          -0.0116/-0.06   -0.0035/+0.17   -0.0019/+0.23-0.0004/+0.21(7)
  every 5th frame                    +0.0023/+0.26   +0.0025/+0.31   +0.0012/+0.31   +0.0006/+0.32
  once a clip                        -0.0207/-0.56   -0.0137/-0.30   -0.0067/+0.05   -0.0031/+0.21
video
  every frame                        +0.0053/+0.66   +0.0049/